# Dental X-ray Analysis for Cavity Detection

This notebook runs the simplified implementation in `dental_xray_detection.py`.

It follows the class labs: Keras Sequential training, CNN/image augmentation, and transfer learning with freeze then fine-tune.

## 1. Setup & Imports

In [ ]:
from pathlib import Path

from dental_xray_detection import (
    BEST_MODEL_PATH,
    OUTPUT_DIR,
    apply_training_augmentation,
    build_augmentation,
    build_model,
    combine_histories,
    ensure_dataset,
    evaluate_model,
    fine_tune,
    load_datasets,
    make_callbacks,
    print_summary,
    save_augmented_preview,
    save_combined_history_plot,
    set_reproducibility,
    train_head,
)

DATASET_PATH = Path("dataset")
BATCH_SIZE = 16
EPOCHS_HEAD = 5
EPOCHS_FINETUNE = 3
WEIGHTS = "imagenet"  # change to "none" for a quick offline smoke test

set_reproducibility()

## 2. Dataset Loading & Preprocessing

In [ ]:
ensure_dataset(DATASET_PATH)
train_dataset, test_dataset, class_names, class_weight = load_datasets(DATASET_PATH, BATCH_SIZE)

## 3. Data Augmentation

In [ ]:
data_augmentation = build_augmentation()
save_augmented_preview(train_dataset, data_augmentation)
augmented_train_dataset = apply_training_augmentation(train_dataset, data_augmentation)
print(f"Saved augmentation preview to {OUTPUT_DIR / 'augmented_samples.png'}")

## 4. Model Architecture - Transfer Learning with MobileNetV2

In [ ]:
dental_model, base_model = build_model(WEIGHTS)

## 5. Phase 1 Training: Classification Head

In [ ]:
callbacks = make_callbacks()
history_phase1 = train_head(
    dental_model,
    augmented_train_dataset,
    test_dataset,
    callbacks,
    EPOCHS_HEAD,
    class_weight,
)

## 6. Phase 2 Fine-Tuning: Last 10 Layers

In [ ]:
history_phase2 = fine_tune(
    dental_model,
    base_model,
    augmented_train_dataset,
    test_dataset,
    callbacks,
    EPOCHS_FINETUNE,
    class_weight,
)

## 7. Model Evaluation

In [ ]:
combined_history = combine_histories(history_phase1, history_phase2)
phase1_epochs = len(history_phase1.history["accuracy"]) if history_phase1 else 0
save_combined_history_plot(combined_history, phase1_epochs)

test_loss, test_accuracy = evaluate_model(test_dataset, class_names)

## 8. Results Summary

In [ ]:
print_summary(combined_history, test_accuracy)
print(f"Saved model: {BEST_MODEL_PATH}")
print(f"Saved figures: {OUTPUT_DIR}")

## 9. Conclusion

The project uses the real TUFTS dental X-ray dataset and a lightweight transfer-learning workflow suitable for CPU training.